In [ ]:
!git clone https://github.com/projectManager22/AraABSAMD
%cd AraABSAMD
!ls

Cloning into 'AraABSAMD'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 250 (delta 58), reused 6 (delta 6), pack-reused 146 (from 1)
Receiving objects: 100% (250/250), 2.04 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/AraABSAMD
AraABSAMD1.xml	AraABSAMD4.xml	AraABSAMD7.xml	license-code  requirements.txt
AraABSAMD2.xml	AraABSAMD5.xml	code		LICENSE.txt
AraABSAMD3.xml	AraABSAMD6.xml	guidelines	README.md


In [ ]:
import glob, xml.etree.ElementTree as ET, pandas as pd

sents, terms = [], []
for f in sorted(glob.glob("AraABSAMD*.xml")):
    root = ET.parse(f).getroot()
    for s in root.findall("sentence"):
        t = s.find("text")
        cat = s.find("aspectCategories/aspectCategory")
        sents.append({"file": f, "id": s.get("id"),
                      "text": (t.text or "") if t is not None else "",
                      "category": cat.get("category"),
                      "polarity": cat.get("polarity")})
        for a in s.findall("aspectTerms/aspectTerm"):
            terms.append({"file": f, "id": s.get("id"),
                          "target": a.get("target"),
                          "polarity": a.get("polarity")})

df_s, df_t = pd.DataFrame(sents), pd.DataFrame(terms)
print("Phrases :", len(df_s))
print("Phrases avec texte :", (df_s.text.str.len() > 0).sum())
print(pd.crosstab(df_s.category, df_s.polarity, margins=True))
print(df_t.polarity.value_counts())

Phrases : 1360
Phrases avec texte : 0
polarity                    Neutral  Positive  conflict  negative  neutral  \
category                                                                     
Curriculum and Instruction        1         0        30       135       55   
Faculty and Staff                 0         0        18       127       49   
Outcomes and Performance          0         0        23       151       15   
Student Experience                0         0        24        91       21   
Technology Integration            0         1         4        10       13   
All                               1         1        99       514      153   

polarity                    positive   All  
category                                    
Curriculum and Instruction       151   372  
Faculty and Staff                105   299  
Outcomes and Performance         104   293  
Student Experience               114   250  
Technology Integration           118   146  
All                   

In [ ]:
import glob, re, xml.etree.ElementTree as ET, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

def clean(t):
    t = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

rows = []
for f in sorted(glob.glob("AraABSAMD*.xml")):
    for s in ET.parse(f).getroot().findall("sentence"):
        cat = s.find("aspectCategories/aspectCategory")
        targets = [a.get("target", "").strip() for a in s.findall("aspectTerms/aspectTerm")]
        targets = list(dict.fromkeys(t for t in targets if t))   # supprime vides et doublons
        rows.append({"text": clean(" ".join(targets)),
                     "category": cat.get("category"),
                     "polarity": cat.get("polarity").lower()})
df = pd.DataFrame(rows)
df = df[df.text.str.len() > 0].reset_index(drop=True)
print(len(df), "exemples")

# Split 70 / 10 / 20, seed 42 (comme l'article)
train, test = train_test_split(df, test_size=0.20, random_state=42, stratify=df.category)
train, val  = train_test_split(train, test_size=0.125, random_state=42, stratify=train.category)
print(len(train), len(val), len(test))

# Baseline TF-IDF + SVM
for task in ["category", "polarity"]:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    clf = LinearSVC().fit(vec.fit_transform(train.text), train[task])
    pred = clf.predict(vec.transform(test.text))
    print("\n=== Baseline", task, "===")
    print("F1 macro :", round(f1_score(test[task], pred, average="macro"), 3))
    print(classification_report(test[task], pred, zero_division=0))

1360 exemples
952 136 272

=== Baseline category ===
F1 macro : 0.577
                            precision    recall  f1-score   support

Curriculum and Instruction       0.51      0.64      0.56        74
         Faculty and Staff       0.66      0.75      0.70        60
  Outcomes and Performance       0.49      0.37      0.42        59
        Student Experience       0.60      0.42      0.49        50
    Technology Integration       0.68      0.72      0.70        29

                  accuracy                           0.57       272
                 macro avg       0.59      0.58      0.58       272
              weighted avg       0.57      0.57      0.57       272


=== Baseline polarity ===
F1 macro : 0.392
              precision    recall  f1-score   support

    conflict       0.00      0.00      0.00        19
    negative       0.58      0.74      0.65        97
     neutral       0.46      0.16      0.24        38
    positive       0.64      0.73      0.68       118


In [ ]:
!pip install -q transformers accelerate

In [ ]:
#les hyperparamètres de l’article (batch 16, learning rate 2e-5, weight decay 0,001, 6 epochs, seed 42)
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
from sklearn.metrics import f1_score, accuracy_score, classification_report

def run(model_name, task, epochs=6):
    set_seed(42)
    labels = sorted(df[task].unique())
    l2i = {l: i for i, l in enumerate(labels)}
    tok = AutoTokenizer.from_pretrained(model_name)

    def prep(d):
        ds = Dataset.from_dict({"text": d.text.tolist(),
                                "label": [l2i[x] for x in d[task]]})
        return ds.map(lambda b: tok(b["text"], truncation=True,
                                    padding="max_length", max_length=64),
                      batched=True)

    tr, va, te = prep(train), prep(val), prep(test)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels))

    def metrics(p):
        pred = p.predictions.argmax(-1)
        return {"f1": f1_score(p.label_ids, pred, average="macro"),
                "acc": accuracy_score(p.label_ids, pred)}

    args = TrainingArguments(
        output_dir="out", per_device_train_batch_size=16,
        per_device_eval_batch_size=32, learning_rate=2e-5, weight_decay=0.001,
        num_train_epochs=epochs, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1",
        seed=42, report_to="none", fp16=torch.cuda.is_available())

    trainer = Trainer(model=model, args=args, train_dataset=tr,
                      eval_dataset=va, compute_metrics=metrics)
    trainer.train()
    out = trainer.predict(te)
    pred = out.predictions.argmax(-1)
    print(f"\n=== {model_name} | {task} ===")
    print("F1 macro :", round(f1_score(out.label_ids, pred, average="macro"), 3))
    print("Accuracy :", round(accuracy_score(out.label_ids, pred), 3))
    print(classification_report(out.label_ids, pred,
                                target_names=labels, zero_division=0))

run("aubmindlab/bert-base-arabertv2", "category")
run("aubmindlab/bert-base-arabertv2", "polarity")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/720k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/952 [00:00<?, ? examples/s]

... (23 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== aubmindlab/bert-base-arabertv2 | polarity ===
F1 macro : 0.397
Accuracy : 0.629
              precision    recall  f1-score   support

    conflict       1.00      0.05      0.10        19
    negative       0.62      0.73      0.67        97
     neutral       0.20      0.05      0.08        38
    positive       0.66      0.82      0.73       118

    accuracy                           0.63       272
   macro avg       0.62      0.41      0.40       272
weighted avg       0.61      0.63      0.58       272



In [ ]:
run("UBC-NLP/ARBERT", "category")
run("UBC-NLP/ARBERT", "polarity")
run("CAMeL-Lab/bert-base-arabic-camelbert-msa", "category")
run("CAMeL-Lab/bert-base-arabic-camelbert-msa", "polarity")

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/952 [00:00<?, ? examples/s]

Map:   0%|          | 0/136 [00:00<?, ? examples/s]

... (58 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== CAMeL-Lab/bert-base-arabic-camelbert-msa | polarity ===
F1 macro : 0.445
Accuracy : 0.673
              precision    recall  f1-score   support

    conflict       0.00      0.00      0.00        19
    negative       0.75      0.74      0.75        97
     neutral       0.36      0.21      0.27        38
    positive       0.69      0.87      0.77       118

    accuracy                           0.67       272
   macro avg       0.45      0.46      0.45       272
weighted avg       0.62      0.67      0.64       272

